# 2. Sistemas de Memoria Avanzados en LangChain

## Objetivos de Aprendizaje
- Comprender las limitaciones de la gestión manual de la memoria.
- Aprender a usar las clases de memoria de LangChain para automatizar el historial de chat.
- Implementar `ConversationBufferMemory` para un historial completo.
- Implementar `ConversationBufferWindowMemory` para limitar el tamaño del historial.
- Implementar `ConversationSummaryMemory` para resumir conversaciones y ahorrar tokens.
- Entender el puente hacia la planificación de agentes.

## El Problema con la Memoria Manual

En el notebook anterior, le dimos memoria a nuestro agente gestionando manualmente una lista `chat_history`. Aunque funcional, este enfoque tiene dos grandes inconvenientes:

1.  **Gestión Tediosa**: Actualizar la lista manualmente después de cada interacción es propenso a errores y no es escalable.
2.  **Límite de Contexto**: Enviar el historial completo en cada llamada consume tokens rápidamente. En conversaciones largas, esto puede exceder el límite de contexto del modelo, provocando errores o un alto costo.

LangChain resuelve esto con un sistema de **clases de Memoria** que automatizan el proceso y ofrecen estrategias para gestionar historiales largos.

### 1. Instalación y Configuración

In [1]:
!pip install langchain langchain-openai openai wikipedia -q

In [2]:
from dotenv import load_dotenv
load_dotenv()
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
os.environ["OPENAI_API_KEY"] = os.getenv("GITHUB_TOKEN", "")
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        base_url=os.environ.get("GITHUB_BASE_URL"),
        api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

✅ LLM de LangChain configurado.


### 2. Agente y Herramientas (Sin Cambios)

Reutilizamos el mismo agente y herramientas. La innovación estará en cómo gestionamos la memoria.

In [3]:
from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langchain_classic import hub

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

prompt = hub.pull("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")

✅ Agente y herramientas listos.


### 3. Automatizando con `ConversationBufferMemory`

Esta es la memoria más básica. Simplemente almacena todos los mensajes en una variable (el "buffer") y los pasa en cada llamada. Replica lo que hicimos manualmente, pero de forma automática.

In [4]:
from langchain_classic.memory import ConversationBufferMemory

# memory_key debe coincidir con la variable de entrada en el prompt ('chat_history')
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

def chat_with_agent_buffer(query: str):
    # Carga el historial de la memoria
    history = memory.load_memory_variables({})["chat_history"]
    
    # Invoca al agente con el historial
    response = agent_executor.invoke({"input": query, "chat_history": history})
    
    # Guarda el nuevo par de pregunta/respuesta en la memoria
    memory.save_context({"input": query}, {"output": response["output"]})
    
    return response["output"]

/var/folders/73/tr7q4l9942176xmsld0751n40000gn/T/ipykernel_72031/1013452071.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


In [5]:
print("Primera pregunta...")
response1 = chat_with_agent_buffer("Háblame del planeta Marte")
print(f"Respuesta 1: {response1}")

print("Segunda pregunta de seguimiento...")
response2 = chat_with_agent_buffer("¿Tiene alguna luna?")
print(f"Respuesta 2: {response2}")

Primera pregunta...


> Entering new AgentExecutor chain...

Invoking: `get_wikipedia_summary` with `{'query': 'Marte'}`


Ocurrió un error: Expecting value: line 1 column 1 (char 0)Parece que hubo un problema al buscar información sobre Marte. Intentaré ayudarte con lo que sé.

Marte es el cuarto planeta del sistema solar en orden de distancia al Sol y el segundo más pequeño después de Mercurio. Es conocido como el "Planeta Rojo" debido a la presencia de óxido de hierro en su superficie, lo que le da un color rojizo característico. Marte ha sido objeto de numerosas misiones espaciales debido a su potencial para albergar vida en el pasado y su similitud con la Tierra en ciertos aspectos, como la duración del día y la presencia de estaciones. Si necesitas más detalles, puedo intentar buscar información nuevamente.

> Finished chain.
Respuesta 1: Parece que hubo un problema al buscar información sobre Marte. Intentaré ayudarte con lo que sé.

Marte es el cuarto planeta del sistema solar 

Como puedes ver, la función `chat_with_agent_buffer` ahora se encarga de cargar y guardar el historial, haciendo el proceso mucho más limpio.

### 4. Gestionando el Contexto con `ConversationBufferWindowMemory`

Para evitar que el historial crezca sin control, podemos usar `ConversationBufferWindowMemory`. Esta memoria solo conserva un número `k` de las últimas interacciones.

Esto es un buen equilibrio entre tener contexto y no sobrecargar al LLM.

In [6]:
from langchain_classic.memory import ConversationBufferWindowMemory

# Creamos una nueva memoria, esta vez con k=1 (solo recuerda el último par de mensajes)
memory_window = ConversationBufferWindowMemory(k=2, memory_key="chat_history", return_messages=True)

def chat_with_agent_window(query: str):
    history = memory_window.load_memory_variables({})["chat_history"]
    response = agent_executor.invoke({"input": query, "chat_history": history})
    memory_window.save_context({"input": query}, {"output": response["output"]})
    return response["output"]

/var/folders/73/tr7q4l9942176xmsld0751n40000gn/T/ipykernel_72031/2240622685.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory_window = ConversationBufferWindowMemory(k=2, memory_key="chat_history", return_messages=True)


In [7]:
print("Pregunta 1: ¿Quién fue Albert Einstein?")
chat_with_agent_window("¿Quién fue Albert Einstein?")

print("Pregunta 2: ¿Y Niels Bohr?")
chat_with_agent_window("¿Y Niels Bohr?")

print("Pregunta 3: ¿De qué trataba la primera pregunta que te hice?")
response3 = chat_with_agent_window("¿De qué trataba la primera pregunta que te hice?")
print(f"Respuesta 3: {response3}")

Pregunta 1: ¿Quién fue Albert Einstein?


> Entering new AgentExecutor chain...

Invoking: `get_wikipedia_summary` with `{'query': 'Albert Einstein'}`


Ocurrió un error: Expecting value: line 1 column 1 (char 0)Parece que hubo un problema al buscar la información. Sin embargo, puedo decirte que Albert Einstein fue un físico teórico alemán, conocido principalmente por desarrollar la teoría de la relatividad, una de las dos grandes teorías fundamentales de la física moderna. Si necesitas más detalles, puedo intentar buscar nuevamente.

> Finished chain.
Pregunta 2: ¿Y Niels Bohr?


> Entering new AgentExecutor chain...

Invoking: `get_wikipedia_summary` with `{'query': 'Albert Einstein'}`


Ocurrió un error: Expecting value: line 1 column 1 (char 0)
Invoking: `get_wikipedia_summary` with `{'query': 'Niels Bohr'}`


Ocurrió un error: Expecting value: line 1 column 1 (char 0)Parece que hubo un problema al buscar información sobre Albert Einstein y Niels Bohr. Sin embargo, puedo proporcion

El agente probablemente no podrá responder a la tercera pregunta, porque con `k=1`, la interacción sobre Einstein ya fue descartada de la memoria para dar paso a la de Niels Bohr.

### 5. Ahorrando Tokens con `ConversationSummaryMemory`

Esta es la estrategia más avanzada. En lugar de guardar los mensajes completos, utiliza un LLM para crear un resumen de la conversación a medida que avanza. En cada nueva interacción, el resumen se actualiza y se pasa como contexto.

Es ideal para conversaciones muy largas donde los detalles específicos se vuelven menos importantes que el contexto general.

In [8]:
from langchain_classic.memory import ConversationSummaryMemory

# Esta memoria necesita un LLM para hacer los resúmenes
memory_summary = ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True)

def chat_with_agent_summary(query: str):
    history = memory_summary.load_memory_variables({})["chat_history"]
    response = agent_executor.invoke({"input": query, "chat_history": history})
    memory_summary.save_context({"input": query}, {"output": response["output"]})
    return response["output"]

/var/folders/73/tr7q4l9942176xmsld0751n40000gn/T/ipykernel_72031/3749718899.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory_summary = ConversationSummaryMemory(llm=llm, memory_key="chat_history", return_messages=True)


In [9]:
print("Pregunta 1: Necesito organizar un viaje a Japón. ¿Cuál es la mejor época para ir?")
chat_with_agent_summary("Necesito organizar un viaje a Japón. ¿Cuál es la mejor época para ir?")

print("Pregunta 2: Suena bien. ¿Qué ciudades me recomiendas visitar en un primer viaje?")
chat_with_agent_summary("¿Qué ciudades me recomiendas visitar en un primer viaje?")

print("Pregunta 3: ¿Sobre qué estábamos hablando?")
response3_summary = chat_with_agent_summary("¿Sobre qué estábamos hablando?")
print(f"Respuesta 3: {response3_summary}")

Pregunta 1: Necesito organizar un viaje a Japón. ¿Cuál es la mejor época para ir?


> Entering new AgentExecutor chain...


La mejor época para visitar Japón depende de tus preferencias, pero generalmente se destacan dos temporadas:

1. **Primavera (marzo a mayo)**: Es una de las épocas más populares debido a la floración de los cerezos (*sakura*), que suele ocurrir entre finales de marzo y principios de abril. El clima es templado y agradable, ideal para explorar ciudades y paisajes.

2. **Otoño (septiembre a noviembre)**: Durante esta temporada, los colores otoñales de los árboles (especialmente los arces) son espectaculares. El clima también es fresco y cómodo, perfecto para actividades al aire libre.

Evita el verano (junio a agosto) si no te gusta el calor y la humedad, aunque es una buena época para festivales. También ten en cuenta que la temporada de tifones ocurre entre agosto y septiembre, lo que podría afectar tus planes.

> Finished chain.
Pregunta 2: Suena bien. ¿Qué ciudades me recomiendas visitar en un primer viaje?


> Entering new AgentExecutor chain...
En un primer viaje a Japón, te recomi

El agente debería poder responder correctamente, ya que la memoria no contiene los mensajes literales, sino un resumen de que el usuario está planeando un viaje a Japón.

## Conclusiones y Transición al Módulo IL2.3

Hemos visto cómo las clases de memoria de LangChain nos permiten crear agentes conversacionales mucho más robustos y eficientes, superando los desafíos de la gestión manual y el tamaño del contexto.

**Resumen de Estrategias de Memoria:**
- **`ConversationBufferMemory`**: Simple y completa. Ideal para conversaciones cortas donde cada detalle importa.
- **`ConversationBufferWindowMemory`**: Eficiente y balanceada. Perfecta para chatbots de servicio al cliente que necesitan contexto reciente.
- **`ConversationSummaryMemory`**: Ahorradora de tokens y escalable. La mejor opción para asistentes de largo plazo y análisis de conversaciones extensas.

Con un agente que puede recordar de manera efectiva, hemos completado una pieza fundamental del rompecabezas. Sin embargo, las conversaciones del mundo real a menudo implican tareas que no se resuelven con una sola herramienta o en un solo paso.

**¿Qué pasa si el agente necesita buscar información, luego calcular algo con esa información y finalmente enviar un correo electrónico?**

Esto requiere que el agente pueda **planificar** una secuencia de acciones. Debe ser capaz de descomponer un objetivo complejo en pasos más pequeños y ejecutarlos en el orden correcto.

Esta es la transición perfecta al **Módulo IL2.3: Planificación y Orquestación de Agentes**, donde exploraremos cómo los agentes pueden crear, seguir y adaptar planes para resolver problemas complejos y multi-paso.